# Track 07 (심화) — SLO & Production Defaults

**구성** : 각 `Session`은 코드 설명(텍스트) -> 코드 -> 해석(텍스트) 순으로 정리되어 있습니다.

**목표** : Track 07의 목표는 SLOSpec 과 production 기본값(타임아웃·동시성·토큰 상한)을 문서화하는 것입니다.

**산출물:** `_out/slo_spec.json`, `_out/recommended_env.md`


In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import logging
import warnings
# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽게 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치 필요: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)
client = exaone.integrations.build_llm_from_env()
print("exaone", exaone.__version__)


**출력 해석:** `model:`·경로 변수가 보이면 이 노트북에서 쓸 client·DATA 가 준비된 것입니다.


## Session 1. `SLOSpec` 예시

각 필드는 알림(alert)·대시보드의 임계값으로 그대로 쓰입니다. 환경마다 값을 조정하세요.


### Session 1-1. 핵심 코드

**하는 일:** 이 단계의 핵심 코드를 실행합니다.

**정상:** 에러 없이 기대 출력이 나옴

**의미:** 각 필드는 알림(alert)·대시보드의 임계값으로 그대로 쓰입니다.


In [ ]:
slo = exaone.observability.SLOSpec(
name="exaone-cookbook-demo",
error_rate_max="0.5%",
availability_min="99.5%",
p95_chat_latency_ms=8000,
p99_chat_latency_ms=15000,
structured_output_success_min="95%",
rag_min_hit_count=1,
notes="Adjust per environment; Track 10 capstone uses this as baseline.",
)
print(json.dumps(slo.to_dict(), ensure_ascii=False, indent=2))


**출력 해석:** 에러 없이 기대 출력이 나옴 이면 이 단계는 통과입니다.

**이상:** 출력이 다르면 위 가이드를 다시 확인하세요.


## Session 2. Production defaults

exaone.observability.production_defaults 는 타임아웃·동시성·토큰의 보수적 기본값 입니다. 부하 테스트 후 환경에 맞게 올리세요.


### Session 2-1. 핵심 코드

**하는 일:** 이 단계의 핵심 코드를 실행합니다.

**정상:** 에러 없이 기대 출력이 나옴

**의미:** exaone.observability.production_defaults 는 타임아웃·동시성·토큰의 보수적 기본값 입니다. 부하 테스트 후 환경에 맞게 올리세요.


In [ ]:
pd = exaone.observability.production_defaults
defaults = {
"RECOMMEND_CONNECT_TIMEOUT_S": pd.RECOMMEND_CONNECT_TIMEOUT_S,
"RECOMMEND_READ_TIMEOUT_S": pd.RECOMMEND_READ_TIMEOUT_S,
"RECOMMEND_MAX_IN_FLIGHT_PER_WORKER": pd.RECOMMEND_MAX_IN_FLIGHT_PER_WORKER,
"RECOMMEND_BATCH_CONCURRENCY": pd.RECOMMEND_BATCH_CONCURRENCY,
"RECOMMEND_MAX_NEW_TOKENS_DEFAULT": pd.RECOMMEND_MAX_NEW_TOKENS_DEFAULT,
}
for k, v in defaults.items():
    print(k, "=", v)


**출력 해석:** 에러 없이 기대 출력이 나옴 이면 이 단계는 통과입니다.

**이상:** 출력이 다르면 위 가이드를 다시 확인하세요.


## Session 3. 산출물 — `slo_spec.json` + `recommended_env.md`


### Session 3-1. 산출물 파일을 디스크에 저장

**하는 일:** 산출물 파일을 디스크에 저장합니다.

**정상:** `saved:`, `saved::` 가 보임

**의미:** 파일로 남겨 두면 다음 Session 이나 회귀 테스트에서 재사용합니다.


In [ ]:
(out_dir / "slo_spec.json").write_text(json.dumps(slo.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

env_lines = [
"# Recommended env (Track 07)",
"",
"Copy into team wiki; values are advisory from `exaone.observability.production_defaults` and `.env.example`.",
"",
"| Variable | Notes |",
"|---|---|",
"| EXAONE_API_KEY | Required for LLM tracks |",
"| CORE_CONTEXT_LENGTH_MAX_TOKENS | Align with model context |",
"| MCP_TOOL_TIMEOUT_S | External tool ceiling |",
"| REQUESTS_CA_BUNDLE / SSL_CERT_FILE | Corp network TLS |",
"",
f"Generated: {datetime.now(timezone.utc).isoformat(timespec='seconds')}",
]
(out_dir / "recommended_env.md").write_text("\n".join(env_lines), encoding="utf-8")
print("saved:", (out_dir / "slo_spec.json").resolve())
print("saved:", (out_dir / "recommended_env.md").resolve())


**출력 해석:** `saved:`, `saved::` 이 보이면 이 단계는 통과입니다.

**이상:** 출력이 다르면 위 가이드를 다시 확인하세요.


## 체크포인트

- [ ] `slo_spec.json` 에 에러율·가용성·p95/p99·구조화 출력 성공률이 모두 명세됨.
- [ ] `recommended_env.md` 에 타임아웃·TLS·컨텍스트 길이 환경변수가 정리됨.

**다음:** Track 08 — Evaluation
